In [10]:
# Kétlépcsős klaszterezés: 
# 1) hibák csoportosítása calc_pattern szerint
# 2) minden calc_pattern csoporton belül optimális klaszterszám kiválasztása (silhouette alapján),
#    majd k-means klaszterezés a question szövegek TF-IDF vektorain

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from pathlib import Path
import textwrap

errors = pd.read_csv('res/e38_18r.csv').query('exact_match == False')
# Kiinduló adatok: a korábban kiszűrt hibás sorok 'errors' DataFrame
assert 'errors' in globals(), "A 'errors' DataFrame-nek már léteznie kell a korábbi lépésből."

def choose_best_k(X, min_k=2, max_k=8, random_state=42):
    """
    Silhouette pontszám alapján választ optimális K-t.
    Korlátok: ha a minta elemszám < 3, K=1.
    Ha n < min_k, visszaad 1-et. Ha hiba, visszaesik 1-re.
    """
    n = X.shape[0]
    if n < 3:
        return 1, None  # túl kicsi a minta
    
    # kandidátus K-k
    max_k = min(max_k, n-1)  # K nem lehet nagyobb, mint n-1
    if max_k < min_k:
        return 1, None
    
    best_k, best_score = None, -np.inf
    for k in range(min_k, max_k+1):
        try:
            km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
            labels = km.fit_predict(X)
            # silhouette az euklideszi térben (TF-IDF normalizált vektorokkal ez működik)
            score = silhouette_score(X, labels, metric='euclidean')
            if score > best_score:
                best_k, best_score = k, score
        except Exception as e:
            # kihagyjuk a hibás eseteket (pl. szinguláris eloszlás)
            continue
    
    if best_k is None:
        return 1, None
    return best_k, best_score

# Eredmény gyűjtők
rows_summary = []
rows_detailed = []

# A TF-IDF vektorizálót minden csoportban külön illesztjük (a belső nyelvi sajátosságok miatt)
grouped = errors.groupby("calc_pattern", dropna=False)


# Futtatjuk a kétlépcsős klaszterezést itt is
summary_rows = []
detailed_rows = []
all_labels = pd.Series(index=errors.index, dtype="Int64", name="inner_cluster")

for pattern, dfp in grouped:
    texts = dfp["question"].fillna("")
    n = len(dfp)
    if n < 3:
        k, sil = 1, None
        labels = np.zeros(n, dtype=int)
    else:
        vect = TfidfVectorizer(stop_words="english")
        X = vect.fit_transform(texts)
        k, sil = choose_best_k(X, min_k=2, max_k=8, random_state=42)
        if k == 1:
            labels = np.zeros(n, dtype=int)
        else:
            km = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = km.fit_predict(X)

    summary_rows.append({
        "calc_pattern": pattern,
        "n_errors": n,
        "chosen_k": int(k),
        "silhouette": None if sil is None else float(sil)
    })

    for (idx, q, lab) in zip(dfp.index, texts, labels):
        detailed_rows.append({
            "index": idx,
            "calc_pattern": pattern,
            "question": q,
            "inner_cluster": int(lab)
        })
        all_labels.loc[idx] = int(lab)

summary_df = pd.DataFrame(summary_rows).sort_values(by="n_errors", ascending=False)
detailed_df = pd.DataFrame(detailed_rows).set_index("index").loc[errors.index]

# Eredmények elmentése
out_dir = Path("res/")
summary_csv = out_dir / "two_stage_cluster_summary.csv"
detailed_csv = out_dir / "two_stage_cluster_detailed.csv"
summary_df.to_csv(summary_csv, index=False)
detailed_df.to_csv(detailed_csv)



# Kimeneti fájlok elérési útjai
summary_csv, detailed_csv


(PosixPath('res/two_stage_cluster_summary.csv'),
 PosixPath('res/two_stage_cluster_detailed.csv'))